# Bedside

> Processing utilities for bedside/ICU data

In [ ]:
#| default_exp bedside

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np, zarr, datetime as dt, warnings, pandas as pd, torch

from pathlib import Path
import datetime as dt

from torch.utils.data import Dataset
from physiojepa.data_preprocessing import calculate_samples_mp, interpolate_nan_clip
from physiojepa.signal import butterworth
from scipy.ndimage import median_filter

## Dataloaders

In [ ]:
#| export
class ForecastingDataset(Dataset):
    def __init__(self, 
                 zarr_files, # zarr files that include samples
                 channels, # channels to use
                 forecast_window_sec, # forecast window (within), suggest 5, 10, 15 minutes
                 outcome_df, # pandas dataframe containing outcomes for zarr files
                 include_labels_in_x=False, # indicator to include y labels in sample data time frame (resampled to the frequency)
                 forecast_within=False, # indicator to create labels when the event occurs within the forecast window. Set to false to only return the last value in the forecast window.
                 y_mapping_column='subject_id', # column mapping corresponding outcome value to zarr file
                 y_date_column='date', # column indicating date of sample collection
                 y_seconds_since_column='Time Stamp (seconds)', # column indicating how many seconds since beginning of waveform
                 y_outcome='hypotension', # outcome column in the y file path
                 max_seq_len_sec=None, # maximum sequence length (in seconds) to use (this is especially relevant when you are returning both stft and raw ts data to keep them in sync)
                 sample_df=None, # dataframe indicating which indices within each zarr file includes a sample
                 sample_seq_len_sec=None, # if no sample_df, generate sequences of this length in seconds as one sample
                 sample_stride_sec=None, #  if no sample_df, seconds of overlap for samples from the same array, if seq_len_seconds == overlap_seconds, there is no overlap
                 frequency=125, # frequency of underlying data
                 butterworth_filters=None, # dictionary of low pass, high pass, and bandpass dictionary to perform on channels
                 median_filter_kernel_size=None, # size of median filter to perform on channels
                 clip_interpolations=None, # dictionary of channels:{'phys_range':..., 'percentiles':...} for filtering and interpolation of filtered values
                 nan_tolerance=0.2, # tolerance for nan values in the data - 0 means no nan allowed, 1 means 100% of nans allowed
                 require_all_channels=True # indicator to require all channels to be present in the sample, if False, will return samples with any of the channels and 0s for the missing channels
                 ):
        if forecast_window_sec % 60 != 0:
            warnings.warn("The forecast_window_sec shoud likely be divisible by 60, since outcomes are typically labeled every 60 sec.")
        self.max_seq_len_sec = max_seq_len_sec
        self.zarr_files = zarr_files
        self.channels = channels
        self.forecast_window_sec = forecast_window_sec
        self.forecast_within = forecast_within
        self.sample_seq_len_sec = sample_seq_len_sec
        self.frequency = frequency
        self.sample_stride_sec = sample_stride_sec
        self.clip_interpolations = clip_interpolations
        self.include_labels_in_x = include_labels_in_x
        self.butterworth_filters = butterworth_filters
        self.median_filter_kernel_size = median_filter_kernel_size
       
        if sample_df is None:
            assert sample_seq_len_sec is not None and sample_stride_sec is not None, "You must provide sample sequence lengths and strides if you do not pass a sample_df"
            print(f"Calculating samples with {sample_seq_len_sec} sec length and {sample_stride_sec} sec stride")
            self.sample_df, _ = calculate_samples_mp(zarr_files, channels=channels, max_seq_len_sec=max_seq_len_sec, sample_seq_len_sec=sample_seq_len_sec, frequency=frequency, stride_sec=sample_stride_sec, include_partial_samples=False, nan_tolerance=nan_tolerance, require_all_channels=require_all_channels)
        else:
            self.sample_df = sample_df.copy()
        
        self.outcome_df = outcome_df.copy()
        self.y_outcome = y_outcome
        self.y_mapping_column = y_mapping_column
        self.y_date_column = y_date_column
        self.y_seconds_since_column = y_seconds_since_column

        self.sample_df[self.y_date_column] = self.sample_df['file'].apply(lambda x: dt.datetime.strptime(Path(x).stem.split('-',  maxsplit=1)[1], '%Y-%m-%d-%H-%M'))
        self.sample_df[self.y_mapping_column] = self.sample_df['file'].apply(lambda x: Path(x).stem.split('-')[0])
        self.sample_df['unique_identifier'] = self.sample_df[self.y_mapping_column].astype(str) + '__' + self.sample_df[self.y_date_column].astype(str)
        self.sample_df[y_seconds_since_column] = (self.sample_df['end_idx'] / self.frequency + self.forecast_window_sec)
        ## round to nearest 60 seconds
        self.sample_df[y_seconds_since_column] = self.sample_df[y_seconds_since_column].apply(lambda x: np.round(x / 60) * 60).astype(float)

        self.outcome_df[self.y_date_column] = pd.to_datetime(self.outcome_df[self.y_date_column])
        self.outcome_df['unique_identifier'] = self.outcome_df[self.y_mapping_column].astype(str) + '__' + self.outcome_df[self.y_date_column].astype(str)
        self.outcome_df = self.outcome_df.loc[self.outcome_df['unique_identifier'].isin(self.sample_df['unique_identifier'].unique())]
        self.outcome_df[y_seconds_since_column] = self.outcome_df[y_seconds_since_column].astype(float)
        self.outcome_df.set_index(['unique_identifier', self.y_seconds_since_column], inplace=True)
        self.outcome_df.sort_index(inplace=True)

        subset_index = pd.MultiIndex.from_arrays([self.sample_df['unique_identifier'], self.sample_df[y_seconds_since_column]])
        mask = subset_index.isin(self.outcome_df.index)
        if not mask.all():
            warnings.warn(f"There are {sum(~mask)} samples in the zarr files that are not in the outcome df, they will be dropped. Sample unique identifiers: {self.sample_df.loc[~mask, 'unique_identifier'].unique().tolist()}")
            self.sample_df = self.sample_df[mask]
        if self.sample_df.empty:
            raise ValueError("There are no samples in the sample_df that match the outcome_df")
        # merge sample and outcome df
        # left = dd.from_pandas(self.sample_df, npartitions=3)
        # right = dd.from_pandas(self.outcome_df, npartitions=3)
        # if not self.forecast_within:
        #     # filter to only exact forecast matches
        #     self.sample_df = dd.merge(left, right, on=['unique_identifier', 'subject_id', 'date', 'Time Stamp (seconds)']).compute()
        # else:
        #     self.sample_df = dd.merge_asof(left, right, on=['unique_identifier', 'subject_id', 'date']).compute()
        self.sample_df.sort_values(by = [y_mapping_column, y_seconds_since_column], ascending=True, inplace=True)
        self.sample_df['start_idx'] = self.sample_df['start_idx'].astype(int)
        self.sample_df['end_idx'] = self.sample_df['end_idx'].astype(int)
        self.total_samples = len(self.sample_df)

    def __len__(self):
        return self.total_samples
    
    def __getitem__(self, idx):
        # get full length x, idx can be a slice
        sample = self.sample_df.iloc[idx]
        root_grp = zarr.open(sample['file'])
        sample_id = sample['unique_identifier'] # grab full ex. p001888__date
        sample_start_seconds = sample['start_idx'] / self.frequency
        sample_end_seconds = sample['end_idx'] / self.frequency
        # forecast_max = sample['end_idx'] + self.forecast_window # indice of max window for forecast
        # forecast_max_seconds = forecast_max / self.frequency # number seconds since beginning of waveform
        
        # Get label time (already rounded to nearest minute in __init__)
        label_time = sample[self.y_seconds_since_column]
        if not self.forecast_within:
            # grab last value in forecast window
            y = self.outcome_df.loc[(sample_id, label_time), self.y_outcome]
        else:
            window_start = sample_end_seconds
            window_end = label_time
            y = self.outcome_df.loc[(sample_id, slice(window_start, window_end)), self.y_outcome].values
            if all(y) == 2:
                y = 2
            elif any(y) == 1:
                y = 1
            else:
                y = 0
        Y = torch.tensor([y], dtype=torch.int64)
        signals = []
        for channel in self.channels:
            if channel in root_grp.array_keys():
                temp = root_grp[channel][sample['start_idx']:sample['end_idx']]
                if self.clip_interpolations is not None and channel in self.clip_interpolations:
                    temp = interpolate_nan_clip(temp, physiological_range_clip=self.clip_interpolations[channel]['phys_range'], percentile_clip=self.clip_interpolations[channel]['percentiles'])
                if self.median_filter_kernel_size is not None:
                    temp = median_filter(temp, size=self.median_filter_kernel_size, mode='nearest')
                if self.butterworth_filters is not None and channel in self.butterworth_filters:
                    freq_range = self.butterworth_filters[channel]
                    btype = 'highpass' if freq_range[0] is None else 'lowpass' if freq_range[1] is None else 'bandpass'
                    freq_range = freq_range[1] if freq_range[0] is None else freq_range[0] if freq_range[1] is None else freq_range
                    temp = butterworth(temp, freq_range=freq_range, btype=btype, fs=self.frequency, order=2)
            else:
                temp = np.zeros((sample['end_idx'] - sample['start_idx'],))
            # if channel == 'ABP':
            #     temp = (temp < 65).astype(np.int32)
            signals.append(temp)
        if self.include_labels_in_x:
            labels_for_sample = self.outcome_df.loc[(sample_id, slice(sample_start_seconds, sample_end_seconds), self.y_outcome)].values
            n_repeats = temp.shape[0]//len(labels_for_sample)
            labels_for_sample = labels_for_sample.repeat(n_repeats) # repeat each element at specified frequency
            signals.append(labels_for_sample)
        X = torch.from_numpy(np.array(signals, dtype=np.float32))
        #sequence_padding_mask = torch.zeros([1, X.shape[-1]]) # channels are all the same length
        return X,Y

In [ ]:
#| export
class SelfSupervisedDataset(Dataset):
    def __init__(self, 
                 zarr_files, # zarr files that include samples
                 channels, # channels to use
                 max_seq_len_sec=None, # maximum sequence length (in seconds) to use (this is especially relevant when you are returning both stft and raw ts data to keep them in sync)
                 sample_df=None, # dataframe indicating which indices within each zarr file includes a sample
                 sample_seq_len_sec=None, # if no sample_df, generate sequences of this length in seconds as one sample
                 sample_stride_sec=None, #  if no sample_df, seconds of overlap for samples from the same array, if seq_len_seconds == overlap_seconds, there is no overlap
                 frequency=125, # frequency of underlying data
                 butterworth_filters=None, # dictionary of low pass, high pass, and bandpass dictionary to perform on channels
                 median_filter_kernel_size=None, # size of median filter to perform on channels
                 clip_interpolations=None, # dictionary of channels:{'phys_range':..., 'percentiles':...} for filtering and interpolation of filtered values
                 nan_tolerance=0.2 # tolerance for nan values in the data - 0 means no nan allowed, 1 means 100% of nans allowed
                 ):
        self.max_seq_len_sec = max_seq_len_sec
        self.zarr_files = zarr_files
        self.channels = channels
        self.sample_seq_len_sec = sample_seq_len_sec
        self.frequency = frequency
        self.sample_stride_sec = sample_stride_sec
        self.clip_interpolations = clip_interpolations
        self.butterworth_filters = butterworth_filters
        self.median_filter_kernel_size = median_filter_kernel_size
       
        if sample_df is None:
            assert sample_seq_len_sec is not None and sample_stride_sec is not None, "You must provide sample sequence lengths and strides if you do not pass a sample_df"
            print(f"Calculating samples with {sample_seq_len_sec} sec length and {sample_stride_sec} sec stride")
            self.sample_df, _ = calculate_samples_mp(zarr_files, channels=channels, max_seq_len_sec=max_seq_len_sec, sample_seq_len_sec=sample_seq_len_sec, frequency=frequency, stride_sec=sample_stride_sec, include_partial_samples=False, nan_tolerance=nan_tolerance)
        else:
            self.sample_df = sample_df.copy()

        self.sample_df['start_idx'] = self.sample_df['start_idx'].astype(int)
        self.sample_df['end_idx'] = self.sample_df['end_idx'].astype(int)
        self.total_samples = len(self.sample_df)

    def __len__(self):
        return self.total_samples
    
    def __getitem__(self, idx):
        # get full length x, idx can be a slice
        sample = self.sample_df.iloc[idx]
        root_grp = zarr.open(sample['file'])
        
        signals = []
        for channel in self.channels:
            temp = root_grp[channel][sample['start_idx']:sample['end_idx']]
            if self.clip_interpolations is not None and channel in self.clip_interpolations:
                temp = interpolate_nan_clip(temp, physiological_range_clip=self.clip_interpolations[channel]['phys_range'], percentile_clip=self.clip_interpolations[channel]['percentiles'])
            if self.median_filter_kernel_size is not None:
                temp = median_filter(temp, size=self.median_filter_kernel_size, mode='nearest')
            if self.butterworth_filters is not None and channel in self.butterworth_filters:
                freq_range = self.butterworth_filters[channel]
                btype = 'highpass' if freq_range[0] is None else 'lowpass' if freq_range[1] is None else 'bandpass'
                freq_range = freq_range[1] if freq_range[0] is None else freq_range[0] if freq_range[1] is None else freq_range
                temp = butterworth(temp, freq_range=freq_range, btype=btype, fs=self.frequency, order=2)
            signals.append(temp)
        
        X = Y = torch.from_numpy(np.array(signals, dtype=np.float32))
        return X,Y

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()